In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/netflix_titles.csv")
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [7]:
print("Shape:", df.shape)
df.info()
print(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate show_id:", df["show_id"].duplicated().sum())
df_raw = df.copy()

Shape: (8807, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB
show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
des

STEP 1 : REMOVAL OF DUPLICATE ROWS

In [16]:

before = len(df)
df = df.drop_duplicates()
df = df.drop_duplicates(subset="show_id")
print("Rows removed:", before - len(df))

8807
Rows removed: 0


STEP 2 : TRIMMING EXTRA SPACES FROM ALL TEXT COLUMNS

In [21]:
#showing all columns having dtype object
txt_cols = df.select_dtypes(include="object").columns
for col in txt_cols:
    df[col] = df[col].str.strip()

STEP 3: FIXING MISPLACED ROWS

In [32]:
#shows rows which are misplaced
df[df["rating"].str.contains("min", na=False)][["title", "rating", "duration"]]
mask = df["rating"].str.contains("min", na=False)
df.loc[mask, "duration"] = df.loc[mask, "rating"]
df.loc[mask, "rating"] = np.nan
#fixes country errors
df["country"] = df["country"].str.replace(r",\s*$", "", regex=True).str.strip()

STEP 4: HANDLING MISSING VALUES

In [33]:
df["director"] = df["director"].fillna("Unknown")
df["cast"]     = df["cast"].fillna("Unknown")
df["country"]  = df["country"].fillna("Unknown")
df["rating"]   = df["rating"].fillna("Not Rated")

STEP 5:SPLITTING DURATION INTO NUMBER AND UNIT

In [34]:
extracted = df["duration"].str.extract(r"(\d+)\s*(\w+)")
df["duration_value"] = extracted[0].astype("Int64")
df["duration_unit"]  = extracted[1].str.replace("Seasons", "Season")
df = df.drop(columns="duration")

STEP 6 : CONVERTED DATE_ADDED INTO REAL DATE

In [35]:
df["date_added"] = pd.to_datetime(df["date_added"], format="%B %d, %Y", errors="coerce")


df["year_added"]  = df["date_added"].dt.year.astype("Int64")
df["month_added"] = df["date_added"].dt.month_name()


df["date_added"] = df["date_added"].dt.strftime("%d-%m-%Y")

In [41]:
print("Shape:", df.shape)
df.info()
print(df.isnull().sum()[df.isnull().sum() > 0])
print("Duplicates:", df.duplicated().sum())
print(df["duration_unit"].value_counts())
df.head()

Shape: (8807, 15)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   show_id         8807 non-null   object
 1   type            8807 non-null   object
 2   title           8807 non-null   object
 3   director        8807 non-null   object
 4   cast            8807 non-null   object
 5   country         8807 non-null   object
 6   date_added      8797 non-null   object
 7   release_year    8807 non-null   int64 
 8   rating          8807 non-null   object
 9   listed_in       8807 non-null   object
 10  description     8807 non-null   object
 11  duration_value  8807 non-null   Int64 
 12  duration_unit   8807 non-null   object
 13  year_added      8797 non-null   Int64 
 14  month_added     8797 non-null   object
dtypes: Int64(2), int64(1), object(12)
memory usage: 1.0+ MB
date_added     10
year_added     10
month_added    10
dtype: int64

,show_id,type,title,director,cast,country,date_added,release_year,rating,listed_in,description,duration_value,duration_unit,year_added,month_added
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,25-09-2021,2020,PG-13,Documentaries,"As her father nears the end of his life, filmm...",90,min,2021,September
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,24-09-2021,2021,TV-MA,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",2,Season,2021,September
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,24-09-2021,2021,TV-MA,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,1,Season,2021,September
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,24-09-2021,2021,TV-MA,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",1,Season,2021,September
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,24-09-2021,2021,TV-MA,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,2,Season,2021,September


In [42]:
from google.colab import files
df.to_csv("netflix_cleaned.csv", index=False)
files.download("netflix_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>